# PPR Score-Threshold Subgraph Exploration

Tests whether **score-thresholding** on absorbed PPR probability gives
naturally size-varied subgraphs across alpha values.

**Why the old conductance-sweep approach failed:** `epsilon=1e-3` stopped the
push algorithm early, so all alphas produced ~1-hop subgraphs regardless.

**New mechanism:**
1. Run push-PPR with tiny `PUSH_EPSILON` (1e-5) — propagate until most mass is absorbed.
2. Select nodes where `absorbed_score(v) > τ` for a fixed threshold τ.
3. Low restart α → mass spreads far → many nodes exceed τ → large subgraph.
4. High restart α → mass absorbed near seed → few nodes exceed τ → small subgraph.

**Convention:** α = classic PageRank *restart* probability (high = local).
Internally `local_ppr.py` uses *continuation* = 1 − α.

| α_restart | continuation | E[depth]=(1−α)/α |
|-----------|-------------|-------------------|
| 0.90      | 0.10        | 0.11              |
| 0.50      | 0.50        | 1.00              |
| 0.25      | 0.75        | 3.00              |
| 0.15      | 0.85        | 5.67              |
| 0.05      | 0.95        | 19.0              |

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
import os
DATASETS         = ['Cora', 'CiteSeer', 'PubMed']
N_PAIRS          = 100
OUT_DIR          = 'exploring'
os.makedirs(OUT_DIR, exist_ok=True)

# Push epsilon: very small so the full score distribution is computed
PUSH_EPSILON     = 1e-5

# Score thresholds τ to sweep
SCORE_THRESHOLDS = [1e-2, 5e-3, 1e-3, 5e-4, 1e-4]

# Classic restart alphas (high = local, low = global)
CLASSIC_ALPHAS   = [0.90, 0.50, 0.25, 0.15, 0.05]

K_HOPS = [1, 2, 3]


In [ ]:
import sys, pickle
sys.path.insert(0, os.path.abspath('.'))
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from torch_geometric.utils import k_hop_subgraph
from torch_geometric.data import Data
from subgrapher.utils.local_ppr import build_sparse_adj, approximate_ppr


In [ ]:
def load_dataset(name):
    """Load PyG processed data without triggering aiohttp download."""
    for root in ['data/Planetoid', 'data/planetoid']:
        proc = os.path.join(root, name, 'processed', 'data.pt')
        if os.path.exists(proc):
            loaded = torch.load(proc, weights_only=False)
            first  = loaded[0] if isinstance(loaded, (list, tuple)) else loaded
            if hasattr(first, 'edge_index'):
                data = first
            elif isinstance(first, dict):
                data = Data(**first)
            else:
                raise RuntimeError(f'Unknown PyG format: {type(first)}')
            if data.num_nodes is None:
                data.num_nodes = int(data.x.size(0))
            return data
    raise FileNotFoundError(
        f'{name} not found in data/Planetoid/ or data/planetoid/. '
        f'Run any notebook that loads it first.')


def bfs_max_dist(adj, seeds, node_set):
    visited = {s: 0 for s in seeds}
    queue   = list(seeds)
    while queue:
        node = queue.pop(0)
        for nb in adj[node].indices:
            if nb not in visited:
                visited[nb] = visited[node] + 1
                queue.append(nb)
    dists = [visited.get(n, 0) for n in node_set if n not in seeds]
    return max(dists) if dists else 0


def score_threshold_nodes(scores, tau, seeds):
    above = set(int(n) for n in np.where(scores > tau)[0])
    return above | seeds


In [ ]:
all_results = {}

for DATASET in DATASETS:
    print(f'\n{'='*55}')
    print(f'  Dataset: {DATASET}')
    print(f'{'='*55}')

    data    = load_dataset(DATASET)
    adj_csr = build_sparse_adj(data.edge_index, data.num_nodes)
    print(f'  Nodes: {data.num_nodes}  Edges: {data.edge_index.size(1)}')

    torch.manual_seed(42)
    perm  = torch.randperm(data.edge_index.size(1))[:N_PAIRS]
    pairs = [(int(data.edge_index[0, i]), int(data.edge_index[1, i])) for i in perm]

    # ── Extract k-hop and raw PPR scores ──────────────────────────────────
    khop_sets = {k: [] for k in K_HOPS}
    ppr_raw   = {ca: [] for ca in CLASSIC_ALPHAS}

    print(f'  Running PPR (PUSH_EPSILON={PUSH_EPSILON}) ...')
    for i, (u, v) in enumerate(pairs):
        if i % 25 == 0:
            print(f'    [{i}/{N_PAIRS}]')
        for k in K_HOPS:
            ids, _, _, _ = k_hop_subgraph(
                [u, v], k, data.edge_index,
                relabel_nodes=False, num_nodes=data.num_nodes)
            khop_sets[k].append(set(ids.tolist()))
        for ca in CLASSIC_ALPHAS:
            scores = approximate_ppr(
                adj_csr, {u, v}, alpha=1.0 - ca, epsilon=PUSH_EPSILON)
            ppr_raw[ca].append(scores)

    # ── Build (alpha, tau) node sets ──────────────────────────────────────
    ppr_sets = {}
    for ca in CLASSIC_ALPHAS:
        for tau in SCORE_THRESHOLDS:
            ppr_sets[(ca, tau)] = [
                score_threshold_nodes(ppr_raw[ca][i], tau, {u, v})
                for i, (u, v) in enumerate(pairs)]

    # ── Statistics ────────────────────────────────────────────────────────
    khop_stats = {k: {'mean_size': np.mean([len(s) for s in khop_sets[k]])}
                  for k in K_HOPS}
    stats = {}
    for ca in CLASSIC_ALPHAS:
        for tau in SCORE_THRESHOLDS:
            ns    = ppr_sets[(ca, tau)]
            sizes = [len(s) for s in ns]
            deps  = [bfs_max_dist(adj_csr, {u, v}, ns[i])
                     for i, (u, v) in enumerate(pairs)]
            stats[(ca, tau)] = {
                'mean_size':  np.mean(sizes),
                'mean_depth': np.mean(deps),
            }

    all_results[DATASET] = {
        'data': data, 'pairs': pairs, 'adj_csr': adj_csr,
        'khop_sets': khop_sets, 'khop_stats': khop_stats,
        'ppr_sets': ppr_sets, 'ppr_raw': ppr_raw,
        'stats': stats,
    }

    mid_tau = SCORE_THRESHOLDS[len(SCORE_THRESHOLDS)//2]
    print(f'\n  Summary at τ={mid_tau}:')
    print(f'  {"α":>6}  {"mean_size":>10}  {"mean_depth":>11}')
    for ca in CLASSIC_ALPHAS:
        s = stats[(ca, mid_tau)]
        print(f'  {ca:>6.2f}  {s["mean_size"]:>10.1f}  {s["mean_depth"]:>11.2f}')
    print(f'  k-hop: ' + '  '.join(f'k={k}: {khop_stats[k]["mean_size"]:.0f}'
                                    for k in K_HOPS))

print('\nAll datasets done.')


In [ ]:
# ── Plots per dataset ────────────────────────────────────────────────────
colors_tau = plt.cm.viridis(np.linspace(0.15, 0.85, len(SCORE_THRESHOLDS)))
colors_hop = ['green', 'orange', 'red']

for DATASET in DATASETS:
    r          = all_results[DATASET]
    stats      = r['stats']
    khop_stats = r['khop_stats']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    for tau, col in zip(SCORE_THRESHOLDS, colors_tau):
        sizes = [stats[(ca, tau)]['mean_size'] for ca in CLASSIC_ALPHAS]
        ax.plot(CLASSIC_ALPHAS, sizes, 'o-', color=col, label=f'τ={tau:.0e}')
    for k, col in zip(K_HOPS, colors_hop):
        ax.axhline(khop_stats[k]['mean_size'], color=col, linestyle='--', alpha=0.6,
                   label=f'{k}-hop ({khop_stats[k]["mean_size"]:.0f})')
    ax.set_xlabel('Classic restart α  (← global | local →)')
    ax.set_ylabel('Mean subgraph size (nodes)')
    ax.set_title('Size vs α  (score > τ)')
    ax.invert_xaxis()
    ax.legend(fontsize=8, ncol=2)

    ax = axes[1]
    for tau, col in zip(SCORE_THRESHOLDS, colors_tau):
        depths = [stats[(ca, tau)]['mean_depth'] for ca in CLASSIC_ALPHAS]
        ax.plot(CLASSIC_ALPHAS, depths, 'o-', color=col, label=f'τ={tau:.0e}')
    theory = [(1 - ca) / ca for ca in CLASSIC_ALPHAS]
    ax.plot(CLASSIC_ALPHAS, theory, 'k--', alpha=0.4, label='Theory (1−α)/α')
    for k, col in zip(K_HOPS, colors_hop):
        ax.axhline(k, color=col, linestyle=':', alpha=0.5, label=f'{k}-hop')
    ax.set_xlabel('Classic restart α  (← global | local →)')
    ax.set_ylabel('Mean max depth from seed')
    ax.set_title('Depth vs α  (score > τ)')
    ax.invert_xaxis()
    ax.legend(fontsize=8, ncol=2)

    plt.suptitle(f'{DATASET}: PPR score-threshold subgraphs', fontsize=12)
    plt.tight_layout()
    out = os.path.join(OUT_DIR, f'{DATASET}_ppr_score_thresh_stats.png')
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved {out}')


In [ ]:
# ── Heatmap per dataset ──────────────────────────────────────────────────
for DATASET in DATASETS:
    stats = all_results[DATASET]['stats']

    size_mat  = np.array([[stats[(ca, tau)]['mean_size']  for tau in SCORE_THRESHOLDS]
                           for ca in CLASSIC_ALPHAS])
    depth_mat = np.array([[stats[(ca, tau)]['mean_depth'] for tau in SCORE_THRESHOLDS]
                           for ca in CLASSIC_ALPHAS])

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    tau_labels   = [f'{t:.0e}' for t in SCORE_THRESHOLDS]
    alpha_labels = [str(ca) for ca in CLASSIC_ALPHAS]

    for ax, mat, title, fmt in [
        (axes[0], size_mat,  'Mean subgraph size',  '.0f'),
        (axes[1], depth_mat, 'Mean max depth',       '.1f'),
    ]:
        im = ax.imshow(mat, aspect='auto', cmap='YlOrRd')
        ax.set_xticks(range(len(SCORE_THRESHOLDS)))
        ax.set_xticklabels(tau_labels, fontsize=9)
        ax.set_yticks(range(len(CLASSIC_ALPHAS)))
        ax.set_yticklabels(alpha_labels, fontsize=9)
        ax.set_xlabel('Score threshold τ  (← strict | loose →)')
        ax.set_ylabel('Classic restart α  (← global | local →)')
        ax.set_title(title)
        plt.colorbar(im, ax=ax)
        for ri in range(mat.shape[0]):
            for ci in range(mat.shape[1]):
                ax.text(ci, ri, f'{mat[ri,ci]:{fmt}}', ha='center', va='center',
                        fontsize=8,
                        color='white' if mat[ri,ci] > mat.max()*0.6 else 'black')

    plt.suptitle(f'{DATASET}: (α × τ) grid', fontsize=12)
    plt.tight_layout()
    out = os.path.join(OUT_DIR, f'{DATASET}_ppr_score_thresh_heatmap.png')
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved {out}')


In [ ]:
# ── Example visualization per dataset ───────────────────────────────────
try:
    import networkx as nx
    mid_tau = SCORE_THRESHOLDS[len(SCORE_THRESHOLDS)//2]

    for DATASET in DATASETS:
        r        = all_results[DATASET]
        data     = r['data']
        pairs    = r['pairs']
        ppr_sets = r['ppr_sets']
        khop_sets= r['khop_sets']

        pair_idx = 0
        u, v     = pairs[pair_idx]

        all_nodes = set([u, v])
        for k in K_HOPS:
            all_nodes |= khop_sets[k][pair_idx]
        for ca in CLASSIC_ALPHAS:
            all_nodes |= ppr_sets[(ca, mid_tau)][pair_idx]

        src_np = data.edge_index[0].numpy()
        dst_np = data.edge_index[1].numpy()
        mask   = np.array([src_np[i] in all_nodes and dst_np[i] in all_nodes
                           for i in range(len(src_np))])
        sub_ei = data.edge_index[:, torch.from_numpy(mask)]

        G = nx.Graph()
        G.add_nodes_from(sorted(all_nodes))
        G.add_edges_from(sub_ei.t().tolist())
        pos = nx.spring_layout(G, seed=42)

        def draw_sub(ax, title, highlight):
            cols = ['red' if n in (u, v) else
                    'dodgerblue' if n in highlight else 'lightgray'
                    for n in G.nodes()]
            nx.draw(G, pos=pos, ax=ax, node_color=cols, node_size=50,
                    with_labels=False, edge_color='lightgray', width=0.4)
            ax.set_title(title, fontsize=8)

        n_cols = len(K_HOPS) + len(CLASSIC_ALPHAS)
        fig, axes = plt.subplots(1, n_cols, figsize=(3.5 * n_cols, 3.5))
        for i, k in enumerate(K_HOPS):
            draw_sub(axes[i], f'{k}-hop  n={len(khop_sets[k][pair_idx])}',
                     khop_sets[k][pair_idx])
        for j, ca in enumerate(CLASSIC_ALPHAS):
            ns = ppr_sets[(ca, mid_tau)][pair_idx]
            draw_sub(axes[len(K_HOPS)+j],
                     f'α={ca}  τ={mid_tau:.0e}  n={len(ns)}', ns)

        patches = [
            mpatches.Patch(color='red',        label='Seed (u,v)'),
            mpatches.Patch(color='dodgerblue', label='In subgraph'),
            mpatches.Patch(color='lightgray',  label='Excluded'),
        ]
        fig.legend(handles=patches, fontsize=8, loc='lower center',
                   ncol=3, bbox_to_anchor=(0.5, -0.05))
        plt.suptitle(f'{DATASET}  Pair {pair_idx}: u={u}, v={v}  τ={mid_tau:.0e}',
                     fontsize=10)
        plt.tight_layout()
        out = os.path.join(OUT_DIR, f'{DATASET}_ppr_score_thresh_example.png')
        plt.savefig(out, dpi=120, bbox_inches='tight')
        plt.show()
        print(f'Saved {out}')
except Exception as e:
    print(f'Visualization skipped: {e}')


In [ ]:
# ── Save per-dataset pkl ─────────────────────────────────────────────────
for DATASET in DATASETS:
    r = all_results[DATASET]
    result = {
        'dataset':          DATASET,
        'seed_pairs':       r['pairs'],
        'classic_alphas':   CLASSIC_ALPHAS,
        'score_thresholds': SCORE_THRESHOLDS,
        'push_epsilon':     PUSH_EPSILON,
        'k_hops':           K_HOPS,
        'khop':             {k: [sorted(s) for s in r['khop_sets'][k]] for k in K_HOPS},
        'ppr_sets':         {str(k): [sorted(s) for s in v]
                             for k, v in r['ppr_sets'].items()},
        'ppr_raw_scores':   {ca: [s.tolist() for s in r['ppr_raw'][ca]]
                             for ca in CLASSIC_ALPHAS},
        'stats':            {str(k): v for k, v in r['stats'].items()},
        'khop_stats':       r['khop_stats'],
        'edge_index':       r['data'].edge_index.cpu(),
        'num_nodes':        r['data'].num_nodes,
    }
    save_path = os.path.join(OUT_DIR, f'{DATASET}_ppr_score_thresh_comparison.pkl')
    with open(save_path, 'wb') as f:
        pickle.dump(result, f)
    print(f'Saved {save_path}')

print()
print('Look at the heatmaps: choose τ where sizes span a useful range.')
print('Then pick 3 α values from that τ column giving small/medium/large subgraphs.')
